<a href="https://colab.research.google.com/github/scostavinicius/lean-agent/blob/tools/lean_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agente provador de teoremas em Lean 4

**Dupla:** VINICIUS COSTA SOARES · MATHEUS VINÍCIUS SILVA FREIRE DE CASTRO

Tarefa da Unidade I — IA Agêntica (2026.2)

## Sobre este notebook

Por enquanto, este notebook só prepara o ambiente. As seções do agente estão vazias e serão preenchidas pela dupla ao longo da semana, uma de cada vez.

Cada célula de código começa com um comentário **Por quê**, explicando o motivo de ela existir. Ao adicionar uma célula nova, mantenham esse hábito: ele ajuda o parceiro a entender o que foi feito e já adianta o relatório.

## 1 — Ambiente

Três coisas precisam funcionar antes de escrever qualquer agente: a biblioteca da disciplina, o Lean e o acesso ao modelo. Rodem as células em ordem. Se todas terminarem sem erro, o ambiente está pronto.

In [1]:
!pip install -q "agentkit @ git+https://github.com/silvaan/agentic-ai"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.7/114.7 kB 2.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 20.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.4 MB/s eta 0:00:00


In [2]:
!curl -sSfL https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y --default-toolchain leanprover/lean4:v4.33.0

import os
os.environ["PATH"] = os.path.expanduser("~/.elan/bin") + os.pathsep + os.environ["PATH"]
!lean --version

info: downloading installer
info: default toolchain set to 'leanprover/lean4:v4.33.0'
info: downloading https://releases.lean-lang.org/lean4/v4.33.0/lean-4.33.0-linux.tar.zst
548.3 MiB / 548.3 MiB (100 %) 160.5 MiB/s ETA:   0 s
info: installing /root/.elan/toolchains/leanprover--lean4---v4.33.0
Lean (version 4.33.0, x86_64-unknown-linux-gnu, commit d8b18978322de05a8f3dba51ef03cf5461676c17, Release)


In [3]:
# confirmar que o Lean verifica uma prova de verdade, antes de
# colocar um agente no meio. Se aparecer só "código de retorno: 0", deu certo.
# Experimente trocar "omega" por "rfl" ou por "sorry" e rodar de novo.
from pathlib import Path
import subprocess

Path("Teste.lean").write_text("""
theorem teste (a b : Nat) : a + b = b + a := by
  omega
""")
resultado = subprocess.run(["lean", "Teste.lean"], capture_output=True, text=True)
print("código de retorno:", resultado.returncode)
print(resultado.stdout + resultado.stderr)

código de retorno: 0



In [4]:
# Usamos o gpt-oss-120bpelo plano gratuito do Groq.
# A tarefa proíbe a chave no notebook. Por isso ela é lida de um segredo do
# Colab chamado GROQ_API_KEY (ícone de chave na barra lateral) ou de uma
# variável de ambiente com esse nome.
from agentkit import LLMAPI

try:
    from google.colab import userdata
    os.environ.setdefault("GROQ_API_KEY", userdata.get("GROQ_API_KEY"))
except ImportError:
    pass  # fora do Colab, a variável de ambiente já deve existir

llm = LLMAPI(
    "openai/gpt-oss-120b",
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
    temperature=0.0,
    max_tokens=2000,
)
print(llm.invoke("Responda apenas: ok"))

TimeoutException: Requesting secret GROQ_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.

In [ ]:
import os
from google.colab import userdata
from agentkit import LLMAPI



os.environ["GEMINI_API_KEY"] = ""

try:
    gemini_key = userdata.get("GEMINI_API_KEY")
except Exception:
    gemini_key = os.environ.get("GEMINI_API_KEY", "")

llm = LLMAPI(
    "gemini-3.5-flash-lite",
    api_key=gemini_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    temperature=0.0,
    max_tokens=2000,
)
print(llm.invoke("Responda apenas: ok"))

ok


# Prompt de comportamento do agente

In [ ]:
def agente_corretor(enunciado: str, prova_atual: str, log_analisado: dict, history: list[dict]) -> str:
    
    """Usa o chat contínuo com histórico para gerar uma nova tentativa baseada no feedback do Lean."""
    
    if log_analisado.get("tipo") == "sorry_pendente":
        feedback_lean = "A prova compilou com sucesso, mas o uso de 'sorry' precisa ser substituído por uma demonstração matemática válida."
    else:
        feedback_lean = f"""
        O Lean falhou com o seguinte erro:
        {log_analisado.get('mensagem_limpa', '')}
        
        Contexto atual (Hipóteses disponíveis):
        {log_analisado.get('hipoteses', '')}
        
        Objetivo que você precisa atingir (⊢):
        {log_analisado.get('objetivo', '')}
        """

    # Mensagem do turno atual detalhando o erro e a tentativa que falhou
    mensagem_turno = f"""
    Estamos tentando provar o teorema: {enunciado}

    A tentativa atual de prova foi:
    ```lean
    {prova_atual}
    ```

    FEEDBACK DO VERIFICADOR (LEAN 4):
    {feedback_lean}

    Instrução: Analise o histórico de tentativas anteriores. Não repita táticas que já falharam. 
    Retorne APENAS o código das táticas que ficam após o "by", sem repetir "theorem", cabeçalhos ou "by".
    """

    # Adiciona a nova mensagem do usuário ao histórico (memória)
    history.append({"role": "user", "content": mensagem_turno})
    
    # Invoca o LLM passando todo o histórico acumulado de conversação
    response = llm.invoke(history)
    
    # Salva a resposta do assistente no histórico
    history.append({"role": "assistant", "content": response})
    
    prompt = f"""
    Você é um assistente especialista em Lean 4.
    Estamos tentando provar o seguinte teorema:
    Teorema: {enunciado}

    =========================================
    FEEDBACK DO VERIFICADOR (LEAN 4):
    =========================================
    {feedback_lean}

    =========================================
    DIRETRIZES DE TÁTICAS (GuiaLean4.md):
    =========================================
    1. Tática `intro`: Usada para introduzir variáveis ou hipóteses no contexto local (implicações/quantificadores). Ex: `intro hp hq`.
    2. Tática `rw` (rewrite): Substitui termos usando igualdades. Use colchetes e `←` para sentido inverso. Ex: `rw [Nat.add_zero]`, `rw [← lema]`.
    3. Tática `induction`: Usada para indução em tipos como Nat. Sintaxe:
       induction n with
       | zero => ...
       | succ n ih => ...
    4. Tática `omega`: Provador automático de aritmética linear para Nat/Int (resolve equações e somas lineares).
    5. Tática `simp`: Simplifica o alvo aplicando regras padrão e lemas marcados. Ex: `simp [Nat.add_comm]`.
    6. Tática `rfl` (reflexivity): Fecha metas identicamente iguais por definição.

    =========================================
    RESTRIÇÕES IMPORTANTES:
    =========================================
    - Não utilize teoremas externos complexos prontos que você não tenha provado. Foque em passos fundamentais usando as táticas acima.
    - Siga estritamente uma resolução linha por linha (passo a passo detalhado).

    =========================================
    ESTADO ATUAL:
    =========================================
    A tentativa atual de prova é:
    ```lean
    {prova_atual}
    ```

    =========================================
    INSTRUÇÕES DE SAÍDA:
    =========================================
    - Pense e gere um plano mental de passos antes de escrever o código.
    - Sua escrita durante as demonstrações e correções deve ser APENAS em código Lean 4 puro e detalhado.
    - Retorne APENAS o código das táticas que ficam após o "by". Não repita a palavra "theorem", o cabeçalho e nem a palavra "by" (as tools já fazem essa inserção).
    - Quando achar uma demonstração valida monte uma saida detalhada explicando passo a passo e o motivo de ter feito cada passo
    """

    response = llm.invoke(prompt)
    texto_limpo = response.replace("```lean", "").replace("```", "").strip()
    
    if texto_limpo.startswith("by"):
        texto_limpo = texto_limpo[2:].strip()
        
    return str(texto_limpo)

# Ferramentas

In [ ]:
from agentkit import tool

In [ ]:
@tool
def get_arquivo_lean(enunciado: str, prova: str, nome_arquivo: str) -> None:
    """Gera arquivo no formato lean com nome definido como {nome_arquivo}.lean."""
    prova_indentada = "\n".join("  " + linha for linha in prova.strip().splitlines())
    codigo = f"theorem alvo {enunciado} := by\n{prova_indentada}\n"
    Path(f"{nome_arquivo}.lean").write_text(codigo, encoding="utf-8")

def ler_arquivo_lean(nome_arquivo: str) -> str:
    """Lê arquivo no formato lean"""
    return Path(f"{nome_arquivo}.lean").read_text(encoding="utf-8")

In [ ]:
@tool   # OBS: Não precisa ser tool
def test_prova(enunciado: str, prova: str, nome_arquivo: str = "Prova") -> tuple[bool, str]:
    """Roda o Lean em `theorem alvo <enunciado> := by <prova>` e devolve (aceita, saída do Lean)."""
    caminho = Path(nome_arquivo)
    if caminho.is_file():
        ler_arquivo_lean(str(caminho))

    get_arquivo_lean(enunciado, prova, nome_arquivo)
    arquivo = nome_arquivo + ".lean"
    resultado = subprocess.run(["lean", arquivo], capture_output=True, text=True)

    saida = resultado.stdout + resultado.stderr
    aceita = resultado.returncode == 0 and "sorry" not in saida and "sorry" not in prova

    return aceita, saida

In [ ]:
@tool   # OBS: creio que não precise existir, já que já existe a função verificar_prova
def resolver_teorema(enunciado: str, prova: str, nome_arquivo: str, saida: str) -> str: 
    """Verifica um teorema em Lean 4 ou aciona o agente corretor se houver 'sorry' ou erro."""
    
    if "sorry" in prova or not saida or "error" in saida:
        log_processado = parsear(saida)
        
        return agente_corretor(enunciado, prova, log_processado)
    else:
        return test_prova(enunciado, prova, nome_arquivo)

In [ ]:
@tool
def check(expression: str) -> str:
    """Roda o Lean em `#check <expression>` e devolve o Type do Lean.

    Em caso de string, use aspas simples seguido de aspas duplas
    Exemplo: '"string"' ou seja quando receber uma palavra como Lean atualize o valor dela para '"Lean"'
    """ 
    codigo = f"#check {expression}\n"

    nome_arquivo = "Check.lean"
    Path(nome_arquivo).write_text(codigo, encoding="utf-8")

    arquivo = nome_arquivo
    resultado = subprocess.run(["lean", arquivo], capture_output=True, text=True)

    saida = resultado.stdout + resultado.stderr
    return saida

In [ ]:

@tool
def ler_guia_taticas() -> str:
    """Retorna o guia rápido de sintaxe e táticas básicas do Lean 4."""
    caminho = Path("GuiaLean4.md")
    if caminho.exists():
        return caminho.read_text(encoding="utf-8")
    return "Guia não encontrado."

import re
@tool   # OBS: Não precisa ser tool
def parsear(log_entrada:str)-> str:
    """
    Analisa a saída do verificador do Lean e extrai de forma limpa 
    o tipo de mensagem, o erro e o Goal State (Meta Atual).
    """
    # Lida caso venha como tupla (False, "texto") ou apenas string
    if isinstance(log_entrada, tuple):
        sucesso, texto_log = log_entrada
    else:
        sucesso = False
        texto_log = str(log_entrada)

    dados_analisados = {
        "sucesso": sucesso,
        "tipo": "desconhecido",
        "mensagem_limpa": "",
        "hipoteses": "",
        "objetivo": ""
    }

    # 1. Verifica se é apenas o aviso clássico de 'sorry'
    if "warning: declaration uses `sorry`" in texto_log:
        dados_analisados["tipo"] = "sorry_pendente"
        dados_analisados["mensagem_limpa"] = "A prova contém 'sorry' e precisa ser completada."
        return dados_analisados

    # 2. Se for erro de tática ou compilação
    if "error:" in texto_log:
        dados_analisados["tipo"] = "erro_compilacao"
        
        # Extrai o Goal State (contexto e meta após o símbolo ⊢)
        # O Lean costuma listar as variáveis e o símbolo ⊢ no final do erro
        match_meta = re.search(r'(.*?)\n\s*([a-zA-Z0-9_\s:]*)\n⊢\s*(.*)', texto_log, re.DOTALL)
        
        if match_meta:
            erro_principal = match_meta.group(1).strip()
            hipoteses = match_meta.group(2).strip()
            objetivo = match_meta.group(3).strip()
            
            dados_analisados["mensagem_limpa"] = erro_principal
            dados_analisados["hipoteses"] = hipoteses
            dados_analisados["objetivo"] = objetivo
        else:
            # Fallback se o formato do erro for diferente
            dados_analisados["mensagem_limpa"] = texto_log.strip()

    return dados_analisados

In [ ]:
import time
def registrar_tentativa_chat(history: list[dict], erro_lean: str) -> str:
    """Adiciona o erro atual do Lean ao histórico, envia para o LLM e recebe a nova tática."""
   
    mensagem_feedback = f"""
    A sua tentativa anterior falhou ou precisa de ajustes. 
    Aqui está a saída/erro do Lean 4:
    ```
    {erro_lean}
    ```
    Corrija a prova gerando apenas as táticas válidas após o 'by'.
    """
    
    history.append({"role": "user", "content": mensagem_feedback})
    
    answer = llm.invoke(history)
    
    history.append({"role": "assistant", "content": answer})
    return answer

def resumir_historico_provas(messages: list[dict]) -> str:
    """Resume as tentativas anteriores do Lean preservando táticas que falharam e os erros gerados."""
    conversacao = "\n".join(f"{m['role']}: {m['content']}" for m in messages)
    
    prompt_resumo = [
        {"role": "user", "content": (
            f"<historico_tentativas>\n{conversacao}\n</historico_tentativas>\n\n"
            "Resuma as tentativas de prova acima em até cinco linhas. "
            "Destaque quais táticas falharam e quais erros o Lean retornou, para que não sejam repetidas."
        )}
    ]
    return llm.invoke(prompt_resumo)

@tool
def verificar_prova(enunciado: str, prova: str, nome_arquivo: str, max_iteracoes: int):
    """Verifica se a prova está correta. Se falhar, usa o histórico e o erro do Lean para refinar iterativamente."""

    aceita, saida = test_prova(enunciado, prova, nome_arquivo)
    
    prova_atual = prova
    iteracao = 0

    while not aceita and iteracao < max_iteracoes:
        iteracao += 1
        
        log_analisado = parsear(saida)
        
        prova_atual = agente_corretor(enunciado, prova_atual, log_analisado, historico_prova)
        
        aceita, saida = test_prova(enunciado, prova_atual, nome_arquivo)
        
        print(f"tentativa {iteracao}")
        print(prova_atual)
        
    return aceita, saida, prova_atual

In [ ]:
print(check("4 + 3"))
print(check("Nat.add"))
print(check('"Lean"'))

print(test_prova("(a b : Nat) : a + b = b + a", "omega"))          # esperado: True
print(test_prova("(a b : Nat) : a + b = b + a", "rfl"))            # esperado: False, com o erro
print(test_prova("(a b : Nat) : a + b = b + a", "sorry"))          # esperado: False

4 + 3 : Nat

Nat.add : Nat → Nat → Nat

"Lean" : String

(True, '')
(False, 'Prova.lean:2:2: error: Tactic `rfl` failed: The left-hand side\n  a + b\nis not definitionally equal to the right-hand side\n  b + a\n\na b : Nat\n⊢ a + b = b + a\n')
(False, 'Prova.lean:1:8: warning: declaration uses `sorry`\n')


In [ ]:
prova_atual = "n sei fazer"
print(verificar_prova("(a b c : Nat) : (a + b) + c = a + (b + c)", prova_atual, "Prova", 10))

tentativa 1
Para provar a associatividade da adição em `Nat`, ou seja, `(a + b) + c = a + (b + c)`, podemos utilizar a tática `omega`, que é altamente especializada e eficiente para resolver problemas de aritmética linear em naturais (`Nat`) e inteiros (`Int`).

Aqui está o código com a tática correta:


omega


### Explicação Passo a Passo:

1. **Análise do Objetivo**: O objetivo atual é provar a igualdade `a + b + c = a + (b + c)` para números naturais (`Nat`), dadas as variáveis `a`, `b` e `c`.
2. **Uso da Tática `omega`**: A tática `omega` implementa o algoritmo de decisão de Presburger para aritmética linear. Como a associatividade da adição de números naturais é uma propriedade estritamente linear e básica, a tática `omega` consegue verificar automaticamente que ambos os lados da equação são equivalentes para quaisquer valores naturais de `a`, `b` e `c`, fechando o objetivo imediatamente sem necessidade de indução manual.
tentativa 2
omega


### Explicação Passo a Passo:

1. **Co

RuntimeError: 429 Too Many Requests: [{
  "error": {
    "code": 429,
    "message": "You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.5-flash-lite\nPlease retry in 44.614941217s.",
    "status": "RESOURCE_EXHAUSTED",
    "details": [
      {
        "@type": "type.googleapis.com/google.rpc.Help",
        "links": [
          {
            "description": "Learn more about Gemini API quotas",
            "url": "https://ai.google.dev/gemini-api/docs/rate-limits"
          }
        ]
      },
      {
        "@type": "type.googleapis.com/google.rpc.QuotaFailure",
        "violations": [
          {
            "quotaMetric": "generativelanguage.googleapis.com/generate_cont

### 2.4 Prompt e contexto

**O que vai aqui:** As instruções que dizem ao modelo como trabalhar.

**Requisito atendido:** prompt e contexto projetados pela dupla.

### 2.5 Mecanismos

**O que vai aqui:** Por exemplo, estado para não repetir tentativas e planejamento da prova em etapas.

**Requisito atendido:** pelo menos 2 mecanismos com função real.

In [ ]:
# Por quê:

### 2.6 Agente

**O que vai aqui:** Juntar modelo, ferramentas e prompt no `Agent` do agentkit.

**Requisito atendido:** o modelo decide quais ferramentas usar e quando parar.

In [ ]:
# Por quê:

### 2.7 Testes e resultados

**O que vai aqui:** Rodar os 10 casos e mostrar entrada, esperado, obtido, aprovado e a taxa de acerto.

**Requisito atendido:** tabela de testes e análise dos erros.

In [ ]:
# Por quê:

## 3 — Relatório (a escrever no fim)

### O que o agente faz

### Por que a tarefa é verificável

### Principais decisões de projeto

### Análise dos erros